In [1]:
import sparknlp
spark = sparknlp.start(apple_silicon=True)
spark

25/08/28 17:17:50 WARN Utils: Your hostname, Js-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.36.130 instead (on interface en0)
25/08/28 17:17:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/jefferyjapheth/.ivy2/cache
The jars for the packages stored in: /Users/jefferyjapheth/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp-silicon_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aec0f4a8-267d-4c0d-9665-3f8eac8b7685;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp-silicon_2.12;6.1.2 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central


:: loading settings :: url = jar:file:/Users/jefferyjapheth/miniconda3/envs/sparknlp/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	found com.github.universal-automata#liblevenshtein;3.0.0 in central
	found com.google.protobuf#protobuf-java-util;3.0.0-beta-3 in central
	found com.google.protobuf#protobuf-java;3.0.0-beta-3 in central
	found com.google.code.gson#gson;2.3 in central
	found it.unimi.dsi#fastutil;7.0.12 in central
	found org.projectlombok#lombok;1.16.8 in central
	found com.google.cloud#google-cloud-storage;2.20.1 in central
	found com.google.guava#guava;31.

In [2]:
# Load the cleaned contract dataset
df = spark.read.parquet("../data/processed/mcc_contracts")

print(f"Dataset loaded! Total records: {df.count():,}")
df.printSchema()

Dataset loaded! Total records: 343,849
root
 |-- contract: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agreement_type: string (nullable = true)
 |-- type_score: string (nullable = true)
 |-- data_split: integer (nullable = true)
 |-- label_count: long (nullable = true)
 |-- type_label: string (nullable = true)



In [3]:
# Step 1: Data Exploration and Basic Stats

# Fix type_score data type
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = df.withColumn("type_score", col("type_score").cast("float"))

# Show class distribution
print("Class Distribution:")
df.groupBy("agreement_type", "type_label").count().orderBy(desc("count")).show()

# Check data splits
print("Data Split Distribution:")
df.groupBy("data_split").count().show()

# Sample records to see the text
print("Sample Records:")
df.select("contract", "description", "agreement_type", "type_score").limit(3).show(truncate=False)

Class Distribution:


+---------------+----------+------+
| agreement_type|type_label| count|
+---------------+----------+------+
|     employment|   LABEL_1|144489|
|       security|   LABEL_0|102328|
|    purchase&ma|   LABEL_4| 47850|
|services&supply|   LABEL_3| 24882|
|    shareholder|   LABEL_5| 15716|
|          lease|   LABEL_2|  8584|
+---------------+----------+------+

Data Split Distribution:
+----------+-----+
|data_split|count|
+----------+-----+
|         1|68736|
|         5|68961|
|         2|68639|
|         3|68807|
|         4|68706|
+----------+-----+

Sample Records:
+-------------------------------+---------------------------------------------------------------+--------------+----------+
|contract                       |description                                                    |agreement_type|type_score|
+-------------------------------+---------------------------------------------------------------+--------------+----------+
|b53262vpexv10w24.txt           |form of amended and r

In [4]:
from sparknlp.base import DocumentAssembler, Finisher
from sparknlp.annotator import Tokenizer, Normalizer, StopWordsCleaner
from pyspark.ml import Pipeline
from pyspark.sql.functions import explode, col, count

In [5]:
# Convert description into Spark NLP document
document_assembler = DocumentAssembler() \
    .setInputCol("description") \
    .setOutputCol("document")

# Tokenize
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("tokens")

# Normalize tokens
normalizer = Normalizer() \
    .setInputCols(["tokens"]) \
    .setOutputCol("normalized_tokens") \
    .setLowercase(True)

# Remove stopwords
stopwords_cleaner = StopWordsCleaner() \
    .setInputCols(["normalized_tokens"]) \
    .setOutputCol("clean_tokens") \
    .setCaseSensitive(False)

# Convert tokens to array of strings
finisher = Finisher() \
    .setInputCols(["clean_tokens"]) \
    .setOutputCols(["finished_tokens"]) \
    .setOutputAsArray(True)

# Build the pipeline
pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    normalizer,
    stopwords_cleaner,
    finisher
])


In [6]:
nlp_model = pipeline.fit(df)
df_tokens = nlp_model.transform(df)

# Check results
df_tokens.select("agreement_type", "finished_tokens").show(200, truncate=False)


+--------------+-------------------------------------------------------------------------------+
|agreement_type|finished_tokens                                                                |
+--------------+-------------------------------------------------------------------------------+
|security      |[form, amended, restated, acquisitioncapital, line, credit]                    |
|security      |[amendment, maturity, extension, agreement, december]                          |
|security      |[exhibit, credit, agreement]                                                   |
|security      |[exhibit, secured, replacement, note, dated, april]                            |
|security      |[securities, purchase, agreement]                                              |
|security      |[waiver, concerning, security, agreement]                                      |
|security      |[amendment, credit, agreement]                                                 |
|security      |[specimen, sub

In [7]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# Explode tokens into separate rows
tokens_df = df_tokens.select(
    "agreement_type", 
    explode("finished_tokens").alias("token")
)

# Count token frequency per agreement_type
token_counts_df = tokens_df.groupBy("agreement_type", "token") \
    .count()

# Rank tokens by frequency per agreement_type
window_spec = Window.partitionBy("agreement_type").orderBy(col("count").desc())

ranked_tokens_df = token_counts_df.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 50) \
    .orderBy("agreement_type", "rank")

ranked_tokens_df.show(100, truncate=False)


+--------------+------------+-----+----+
|agreement_type|token       |count|rank|
+--------------+------------+-----+----+
|employment    |agreement   |77555|1   |
|employment    |plan        |38018|2   |
|employment    |stock       |34354|3   |
|employment    |employment  |33167|4   |
|employment    |form        |24806|5   |
|employment    |amendment   |18047|6   |
|employment    |incentive   |16842|7   |
|employment    |option      |15209|8   |
|employment    |amended     |14098|9   |
|employment    |executive   |12607|10  |
|employment    |ex          |12386|11  |
|employment    |restricted  |11884|12  |
|employment    |award       |11347|13  |
|employment    |compensation|11332|14  |
|employment    |restated    |10803|15  |
|employment    |dated       |10738|16  |
|employment    |exhibit     |10008|17  |
|employment    |inc         |8507 |18  |
|employment    |letter      |7155 |19  |
|employment    |unit        |6665 |20  |
|employment    |severance   |6102 |21  |
|employment    |

In [8]:
# Pivot to see top tokens per agreement_type side by side
pivot_df = ranked_tokens_df.groupBy("rank") \
    .pivot("agreement_type") \
    .agg(first("token")) \
    .orderBy("rank")

pivot_df.show(truncate=False)


+----+------------+-----------+-----------+-----------+---------------+------------+
|rank|employment  |lease      |purchase&ma|security   |services&supply|shareholder |
+----+------------+-----------+-----------+-----------+---------------+------------+
|1   |agreement   |lease      |agreement  |agreement  |agreement      |agreement   |
|2   |plan        |agreement  |purchase   |amendment  |amendment      |rights      |
|3   |stock       |amendment  |dated      |dated      |dated          |registration|
|4   |employment  |dated      |plan       |credit     |services       |dated       |
|5   |form        |ex         |merger     |note       |consulting     |form        |
|6   |amendment   |first      |amendment  |purchase   |amended        |voting      |
|7   |incentive   |office     |sale       |loan       |management     |amended     |
|8   |option      |sublease   |asset      |amended    |ex             |amendment   |
|9   |amended     |exhibit    |license    |restated   |restated  

In [9]:
from pyspark.sql import functions as F

# Step 1: Count in how many agreement types each token appears
token_type_counts = ranked_tokens_df.groupBy("token") \
    .agg(F.countDistinct("agreement_type").alias("type_count"))

# Step 2: Join back with original ranked_tokens_df
token_uniqueness_df = ranked_tokens_df.join(token_type_counts, on="token", how="left") \
    .withColumn("uniqueness", 
                F.when(F.col("type_count") == 1, F.lit("unique"))
                 .otherwise(F.lit("common"))
               ) \
    .select("agreement_type", "token", "count", "rank", "uniqueness") \
    .orderBy("agreement_type", "rank")

# Step 3: Show the result
token_uniqueness_df.show(100, truncate=False)


+--------------+------------+-----+----+----------+
|agreement_type|token       |count|rank|uniqueness|
+--------------+------------+-----+----+----------+
|employment    |agreement   |77555|1   |common    |
|employment    |plan        |38018|2   |common    |
|employment    |stock       |34354|3   |common    |
|employment    |employment  |33167|4   |unique    |
|employment    |form        |24806|5   |common    |
|employment    |amendment   |18047|6   |common    |
|employment    |incentive   |16842|7   |unique    |
|employment    |option      |15209|8   |common    |
|employment    |amended     |14098|9   |common    |
|employment    |executive   |12607|10  |unique    |
|employment    |ex          |12386|11  |common    |
|employment    |restricted  |11884|12  |unique    |
|employment    |award       |11347|13  |unique    |
|employment    |compensation|11332|14  |unique    |
|employment    |restated    |10803|15  |common    |
|employment    |dated       |10738|16  |common    |
|employment 

In [10]:
from pyspark.sql.functions import explode, col, regexp_extract_all

# Example: collect all top tokens into a single pattern per agreement_type
token_list_df = tokens_df.groupBy("agreement_type") \
    .agg(F.collect_list("token").alias("tokens"))

# Convert list to a single regex pattern
token_list_df = token_list_df.withColumn(
    "pattern",
    F.concat_ws("|", "tokens")  # token1|token2|...
)

# Join to main DF
df_pattern = df.join(token_list_df, on="agreement_type")

# Use Spark SQL regex function
df_matches = df_pattern.withColumn(
    "matched_tokens",
    F.expr("regexp_extract_all(lower(description), pattern)")
)

# Explode and count frequencies
phrase_counts_df = df_matches.select(
    "agreement_type", explode("matched_tokens").alias("token")
).groupBy("agreement_type", "token") \
 .count() \
 .orderBy("agreement_type", col("count").desc())


In [11]:
from pyspark.sql import functions as F
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import Tokenizer, Normalizer, StopWordsCleaner
from pyspark.ml import Pipeline

# 1. Prepare the text column
document_assembler = DocumentAssembler() \
    .setInputCol("description") \
    .setOutputCol("document")

tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

normalizer = Normalizer() \
    .setInputCols(["token"]) \
    .setOutputCol("normalized") \
    .setLowercase(True)

stop_cleaner = StopWordsCleaner() \
    .setInputCols(["normalized"]) \
    .setOutputCol("cleanTokens") \
    .setCaseSensitive(False)

# Pipeline
pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    normalizer,
    stop_cleaner
])

processed_df = pipeline.fit(df).transform(df)

# 2. Explode tokens
tokens_df = processed_df.select(
    "agreement_type",
    F.explode("cleanTokens.result").alias("token")
)

# 3. Count token frequency per type
freq_df = tokens_df.groupBy("agreement_type", "token").count()

# 4. Compute global frequency
global_df = freq_df.groupBy("token").agg(F.sum("count").alias("global_count"))

# 5. Join to compute uniqueness score
uniqueness_df = freq_df.join(global_df, on="token") \
    .withColumn("uniqueness", F.col("count") / F.col("global_count")) \
    .orderBy(F.desc("uniqueness"))

# 6. Get top N unique tokens per agreement_type
from pyspark.sql.window import Window

window = Window.partitionBy("agreement_type").orderBy(F.desc("uniqueness"))
top_keywords_df = uniqueness_df.withColumn("rank", F.row_number().over(window)) \
    .filter(F.col("rank") <= 5) \
    .select("agreement_type", "token", "count", "global_count", "uniqueness", "rank")

top_keywords_df.show(100, truncate=False)


+---------------+----------------+-----+------------+----------+----+
|agreement_type |token           |count|global_count|uniqueness|rank|
+---------------+----------------+-----+------------+----------+----+
|employment     |melendres       |1    |1           |1.0       |1   |
|employment     |wilbourn        |1    |1           |1.0       |2   |
|employment     |requirement     |19   |19          |1.0       |3   |
|employment     |rajat           |5    |5           |1.0       |4   |
|employment     |agreementkenneth|7    |7           |1.0       |5   |
|lease          |leggat          |1    |1           |1.0       |1   |
|lease          |creekview       |1    |1           |1.0       |2   |
|lease          |togethersoft    |1    |1           |1.0       |3   |
|lease          |alexandria      |1    |1           |1.0       |4   |
|lease          |wow             |1    |1           |1.0       |5   |
|purchase&ma    |meemic          |1    |1           |1.0       |1   |
|purchase&ma    |com

In [12]:
spark.stop()